In [1]:
#Sheet Pvt PO
import pandas as pd

PO_df = pd.read_excel("PTC draft.xlsx")

In [2]:
# ======================================================
# 1. Build base PN–Order table
# ======================================================
df = (
    PO_df[['PN AL78', 'Order No']]
    .astype({'PN AL78': str})  
    .drop_duplicates(subset=['PN AL78', 'Order No'])  # 👈 KEY LINE
    .sort_values(
        by=['PN AL78', 'Order No'],
        ascending=[True, True]
    )
    .reset_index(drop=True)
)



In [3]:
# ======================================================
# 2. Line show (count per PN)
# ======================================================
df['Line show'] = df.groupby('PN AL78')['Order No'].transform('count')

In [4]:
# ======================================================
# 3. chk = "chg" only on first row of each PN
# ======================================================
df['chk'] = ''
df.loc[df.groupby('PN AL78').head(1).index, 'chk'] = 'chg'

In [5]:
# ======================================================
# 4. PO = comma-separated Order No (first row only)
# ======================================================
po_map = (
    df.groupby('PN AL78')['Order No']
    .apply(lambda x: ','.join(x.astype(str)))
)

df['PO'] = ''
first_idx = df.groupby('PN AL78').head(1).index
df.loc[first_idx, 'PO'] = df.loc[first_idx, 'PN AL78'].map(po_map)


In [6]:
# ======================================================
# 5. To row (Excel row reference of LAST occurrence)
# ======================================================

# Get last index per PN AL78
last_row_map = (
    df.groupby('PN AL78')
      .apply(lambda x: x.index.max())
)

# Build Excel row reference (B + row number)
to_row_map = last_row_map.astype(int).add(2).astype(str).radd('B')

df['To row'] = ''
df.loc[first_idx, 'To row'] = df.loc[first_idx, 'PN AL78'].map(to_row_map)

C:\Users\Brandon\AppData\Local\Temp\ipykernel_16776\3741949753.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.index.max())


In [7]:
# ======================================================
# 6. Insert 2 empty columns after Order No
# ======================================================
df.insert(2, '', '')
df.insert(3, ' ', '')

In [8]:
# ======================================================
# 7. RIGHT SIDE SUMMARY TABLE
# ======================================================
summary_df = (
    df[['PN AL78', 'Line show']]
    .drop_duplicates()
    .rename(columns={'Line show': 'Lines'})
    .assign(Lines=lambda x: x['Lines'].astype('Int64'))
    .reset_index(drop=True)
)

In [9]:
# Create spacer columns between tables
spacer = pd.DataFrame({'': [''] * len(df)})

# Pad summary_df to same length as df
summary_padded = summary_df.reindex(range(len(df)))

# Combine everything
final_df = pd.concat([df, spacer, summary_padded], axis=1)


In [10]:
final_df.to_excel("Pvt PO draft.xlsx", index=False)
